Q1-1 코드

In [1]:
import csv
import json
import logging

logging.basicConfig(level=logging.INFO)

def make_report(csv_path: str, json_path: str) -> int:
    success_count: int = 0
    results: list[dict] = []
    
    try:
        with open(csv_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                # CSV 헤더나 값에 있을 수 있는 공백 제거
                clean_row: dict[str, str] = {k.strip(): v.strip() for k, v in row.items()}
                
                name: str = clean_row["이름"]
                student_id: str = clean_row["학번"]
                m_str: str = clean_row["중간"]
                f_str: str = clean_row["기말"]
                a_str: str = clean_row["과제"]
                
                # 결측값(빈 문자열) 확인
                if m_str == "" or f_str == "" or a_str == "":
                    scores: dict[str, int | None] = {
                        "중간": int(m_str) if m_str != "" else None,
                        "기말": int(f_str) if f_str != "" else None,
                        "과제": int(a_str) if a_str != "" else None
                    }
                    avg = None
                    grade = None
                    logging.info(f"{name}: 결측값이 있어 성적을 산출할 수 없습니다.")
                else:
                    mid: int = int(m_str)
                    fin: int = int(f_str)
                    asn: int = int(a_str)
                    scores: dict[str, int | None] = {"중간": mid, "기말": fin, "과제": asn}
                    
                    avg: float = mid * 0.3 + fin * 0.5 + asn * 0.2
                    
                    if avg >= 90:
                        grade = "A"
                    elif avg >= 80:
                        grade = "B"
                    elif avg >= 70:
                        grade = "C"
                    else:
                        grade = "F"
                    
                    logging.info(f"{name}: 평균 {avg}, 등급 {grade}")
                
                results.append({
                    "이름": name,
                    "학번": student_id,
                    "점수": scores,
                    "평균": avg,
                    "등급": grade
                })
                success_count += 1
                
    except FileNotFoundError:
        logging.warning(f"파일이 존재하지 않습니다: {csv_path}")
        return 0
    except UnicodeDecodeError:
        logging.error(f"인코딩이 잘못되었습니다: {csv_path}")
        return 0
        
    # JSON 파일 쓰기
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=4)
        
    return success_count

if __name__ == "__main__":
    student_count = make_report("scores.csv", "report.json")
    print(f"\n총 {student_count}명의 데이터를 성공적으로 처리했습니다.")

INFO:root:김언어: 평균 89.5, 등급 B
INFO:root:이국문: 평균 84.4, 등급 B
INFO:root:박영문: 평균 93.5, 등급 A
INFO:root:최역사: 결측값이 있어 성적을 산출할 수 없습니다.



총 4명의 데이터를 성공적으로 처리했습니다.


Q1-2 실행결과

>>> make_report("scores.csv", "report.json")
INFO:root:김언어: 평균 89.5, 등급 B
INFO:root:이국문: 평균 84.4, 등급 B
INFO:root:박영문: 평균 93.5, 등급 A
INFO:root:최역사: 결측값이 있어 성적을 산출할 수 없습니다.
4

>>> # 생성된 report.json의 전체 내용
[
    {
        "이름": "김언어",
        "학번": "2026-10000",
        "점수": {
            "중간": 85,
            "기말": 92,
            "과제": 90
        },
        "평균": 89.5,
        "등급": "B"
    },
    {
        "이름": "이국문",
        "학번": "2026-12345",
        "점수": {
            "중간": 78,
            "기말": 88,
            "과제": 85
        },
        "평균": 84.4,
        "등급": "B"
    },
    {
        "이름": "박영문",
        "학번": "2026-13579",
        "점수": {
            "중간": 95,
            "기말": 90,
            "과제": 100
        },
        "평균": 93.5,
        "등급": "A"
    },
    {
        "이름": "최역사",
        "학번": "2025-11111",
        "점수": {
            "중간": null,
            "기말": 82,
            "과제": 88
        },
        "평균": null,
        "등급": null
    }
]

Q1-3 설명

-csv.DictReader로 데이터를 읽을 때 점수 값이 들어갈 자리가 빈 문자열("")인지 조건문으로 확인
-빈 문자열이 하나라도 발견되면 평균과 등급 변수에 파이썬의 None을 할당하여 json.dump 수행 시 자동으로 JSON 표준인 null로 변환되도록 처리

-인코딩 선택의 근거: 문제에 제시된 원본 데이터(scores.csv)가 한글을 포함하므로 파일 입출력 과정에서 데이터가 깨지지 않도록 utf-8을 명시
추가로 JSON 저장 시 ensure_ascii=False를 설정하여 유니코드 이스케이프 문자(\uXXXX)가 아닌 사람이 읽기 쉬운 한글 원형 그대로 보존되도록 작성

Q2-1 코드

In [2]:
# (a) 사용자 정의 예외 클래스 정의
class InvalidJamoError(ValueError):
    pass

# (b) 자모 분류 함수 구현
def classify_jamo(c: str) -> str:
    # 1. 타입 검사
    if not isinstance(c, str):
        raise TypeError("입력값은 문자열(str)이어야 합니다.")
    
    # 2. 길이 검사 (InvalidJamoError가 아닌 일반 ValueError)
    if len(c) != 1:
        raise ValueError("입력된 문자열의 길이는 1이어야 합니다.")
    
    code = ord(c)
    
    # 3. 유니코드 범위 검사
    # 한글 자음 (ㄱ-ㅎ)
    if 0x3131 <= code <= 0x314E:
        return "자음"
    # 한글 모음 (ㅏ-ㅣ)
    elif 0x314F <= code <= 0x3163:
        return "모음"
    # 그 외 한 글자
    else:
        raise InvalidJamoError(f"'{c}'은(는) 유효한 한글 자모가 아닙니다.")

# (c) 리스트 순회 및 예외 처리
inputs = ["ㄱ", "ㅏ", "가", " ", "AB", 5, "ㅎ", "ㅣ"]

for item in inputs:
    try:
        result = classify_jamo(item)
        print(f"'{item}' -> {result}")
    
    # 예외 처리: 자식 예외인 InvalidJamoError를 먼저 잡아야 함
    except TypeError as e:
        print(f"[TypeError] {e}")
    except InvalidJamoError as e:
        print(f"[InvalidJamoError] {e}")
    except ValueError as e:
        print(f"[ValueError] {e}")

'ㄱ' -> 자음
'ㅏ' -> 모음
[InvalidJamoError] '가'은(는) 유효한 한글 자모가 아닙니다.
[InvalidJamoError] ' '은(는) 유효한 한글 자모가 아닙니다.
[ValueError] 입력된 문자열의 길이는 1이어야 합니다.
[TypeError] 입력값은 문자열(str)이어야 합니다.
'ㅎ' -> 자음
'ㅣ' -> 모음


Q2-2 실행결과

'ㄱ' -> 자음
'ㅏ' -> 모음
[InvalidJamoError] '가'은(는) 유효한 한글 자모가 아닙니다.
[InvalidJamoError] ' '은(는) 유효한 한글 자모가 아닙니다.
[ValueError] 입력된 문자열의 길이는 1이어야 합니다.
[TypeError] 입력값은 문자열(str)이어야 합니다.
'ㅎ' -> 자음
'ㅣ' -> 모음

Q2-3 설명

Exception이 아닌 ValueError를 상속받은 이유: 입력받은 인자가 올바른 타입(문자열)이지만 그 '값(Value)'이 자모 분류라는 목적에 부합하지 않기 때문
 최상위 예외인 Exception을 직접 상속받으면 다른 치명적인 시스템 오류들과 구분하기 어려워지므로 성격에 맞는 내장 예외(ValueError)의 자식으로 만들어 예외의 의미를 명확히 함

예외 처리 순서: InvalidJamoError가 ValueError를 상속받았기 때문에 except 블록을 작성할 때 자식 클래스인 InvalidJamoError를 먼저 잡음
 만약 부모인 ValueError를 먼저 적으면, 다형성에 의해 InvalidJamoError도 부모 쪽으로 빨려 들어가서 잘못된 메시지가 출력

본 과제를 수행할 때 사용한 Ai와의 대화는 assignments 폴더의 ai_used_prompt.txt에 첨부함